In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl

In [ ]:
def load1dFigS1(i):
    return np.loadtxt(f'../20221020 HMIA 13 Hall Transparency/data/{i}/data.tsv')


def load2dFigS1(i, num):
    tmp = np.loadtxt(f'../20221020 HMIA 13 Hall Transparency/data/{i}/data.tsv')
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
def load1dFig5(i):
    return np.loadtxt(f'../20221123 HMIA 13-3 Kondo/data/{i}/data.tsv')


def load2dFig5(i, num):
    tmp = np.loadtxt(f'../20221123 HMIA 13-3 Kondo/data/{i}/data.tsv')
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

In [ ]:
figS1 = plt.figure(figsize=(16, 8),constrained_layout=True)
gs = figS1.add_gridspec(2, 4, width_ratios=(1,1,1,1))
# gs = GridSpec(3, 3, figure=fig1)
#fS1_ax1 = figS1.add_subplot(gs[:2, :1])
fS1_ax2 = figS1.add_subplot(gs[:, :2])
fS1_ax3 = figS1.add_subplot(gs[:1, 2:])
fS1_ax4 = figS1.add_subplot(gs[1, 2:])
#fS1_ax5 = figS1.add_subplot(gs[1, 1:3])


#fS1_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=fS1_ax1.transAxes)
fS1_ax2.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=fS1_ax2.transAxes)
fS1_ax3.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=fS1_ax3.transAxes)
fS1_ax4.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=fS1_ax4.transAxes)
#fS1_ax5.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=fS1_ax5.transAxes)

#fS1_ax1.set_axis_off()
#fS1_ax5.set_axis_off()

##############################

dat = load1dFigS1(5)
nsq = 2
B = dat[:, 1]
vxy1 = -dat[:, 2]
vxy2 = -dat[:, 4]
vxx1 = dat[:, 6]
vxx2 = dat[:, 8]
curr = dat[:, 10]

fS1_ax2.plot(B, (vxy1 / curr/25813), 'b')
fS1_ax2.yaxis.label.set_color('blue')
fS1_ax2.tick_params(axis='y', colors='blue')
fS1_ax2.set_yticks([1/i for i in range(2,9)])
fS1_ax2.set_yticklabels('1/' + str(i) for i in range(2,9))

fS1_ax2.set_ylabel(r'R$_{xy}$ ($h/e^2$)')

ax2 = fS1_ax2.twinx()
ax2.plot(B, (vxx1 / curr), color='red')
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
ax2.set_ylabel(r'R$_{xx}$ ($\Omega$)')

fS1_ax2.set_xlabel("B (T)")
fS1_ax2.grid(lw='0.4', ls='--', color='gray')

# plt.title("Ungated Hall Bar")

a, b = np.polyfit(B[:201], (vxy1/curr)[:201], 1)
density = 1/(1.6e-19*a)
print("Density in 1e11 cm^-2: " + str(density/1e15))
mobility = 1/((vxx1/curr)[100]/nsq) / (density*1.6e-19)
print("Mobility in 1e6 cm^2/Vs: " + str(mobility/100))

dat = load1dFigS1(7)

B = dat[:, 1]
vxy1 = -dat[:, 2]
vxx1 = dat[:, 4]
curr = curr[-1]

fS1_ax2.plot(B, (vxy1 / curr/25813), color='blue')
ax2.plot(B, (vxx1 / curr), color='red')

textstr = '\n'.join((
    r'$n=%.2f \times 10^{11}$ cm$^{-2}$' % (density/1e15, ),
    r'$\mu=%.2f \times 10^{6}$ cm$^2$/Vs' % (mobility/100, )))

fS1_ax2.text(0.05, 0.95, textstr, transform=fS1_ax2.transAxes, fontsize=12,
        verticalalignment='top')
# plt.tight_layout()


fS1_ax2.grid(ls='--', lw=0.4)
fS1_ax2.set_xlabel('B (T)')
fS1_ax2.set_ylabel('G$(e^2/h)$')
# fS_ax2.set_yscale('linear')
# # ax[0].set_xlim(-3.5, -2.4)
# fS_ax2.set_ylim(-0.01, 0.12)
###############################

dat = load2dFigS1(4, 51)
nsq = 2
mag = dat['1']
vg = dat['2']
vxy1 = -dat['4']
vxy2 = -dat['6']
vxx1 = dat['8']
vxx2 = dat['10']
curr = dat['12']

# fig, ax = plt.subplots(1, 2, figsize=(8, 4))
numvg = vg.shape[0]
densities = np.zeros((numvg, 2))
mobilities = np.zeros((numvg, 2))

for j, vxy in enumerate([vxy1, vxy2]):
    for i in range(numvg):
        a, b = np.polyfit(mag[i, :], (vxy/curr)[i, :], 1)
        densities[i, j] = 1/(1.6e-19*a)
    
fS1_ax3.plot(vg[2:, 0], np.mean(densities[2:, :], axis=1) / 1e15, "b.")# label='Vxy'+str(j+1))
# ax[0].legend()
fS1_ax3.set_xlabel("$V_g (V)$")
fS1_ax3.set_ylabel(r"Density (1e11 cm$^{-2}$)")

a, b = np.polyfit(vg[2:, 0], densities[2:, 0] / 1e15, 1)
capperarea = a * (1.6e-19) * 1e11
print(capperarea)
# ax[0].plot(np.linspace(-1, 0.1, 10), a*np.linspace(-1, 0.1, 10) + b, '--', lw=1, color='tab:orange', label='Capacitance fit')
# fS1_ax3.legend()

for j, vxx in enumerate([vxx1, vxx2]):
    for i in range(numvg):
        mobilities[i, j] = 1/((vxx/curr)[i, 50]/nsq) / (densities[i, 0]*1.6e-19)
    # ax[1].plot(densities[2:, 0] / 1e4, mobilities[2:, j] * 1e4, ".", label='Vxx'+str(j+1))
fS1_ax4.plot(densities[2:, 0] / 1e15, np.mean(mobilities[2:, :], axis=1) * 1e4 * 1e-6, "b.")#, label='Vxx'+str(j+1))
a, b = np.polyfit(np.log(densities[2:30,0]), np.log(np.mean(mobilities[2:30, :], axis=1)), 1)
fS1_ax4.plot(densities[2:, 0] / 1e15, np.exp(b) * (densities[2:, 0]**a) * 1e4 * 1e-6, "r--" , label='$\mu = n^{0.4}$')

fS1_ax4.legend()
fS1_ax4.set_xlabel(r'Density ($10^{11}$ cm$^{-2}$)')
fS1_ax4.set_ylabel(r'Mobility ($10^6$ cm$^2$/Vs)')

for axi in [fS1_ax3, fS1_ax4]:
    axi.grid(ls='--', lw=0.4)
plt.tight_layout()

plt.savefig("FigureS1_noNEMO.pdf")